# MCP server 4 — Work orders

Explore the complete read-only work-order MCP contract: discovery, filtering, record details, tasks, costs, plan variance, KPIs, schedules, technician assignments, and failure codes.

**Tutorial contract:** run the tutorial notebooks in order and execute this notebook from top to bottom. The shared CouchDB infrastructure check is demonstrated in notebook 02. This notebook validates its own work-order data dependency through real MCP queries and never exposes or calls a write tool.


In [ ]:
from pathlib import Path
import json, os, sys

def find_repo(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "servers").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the AssetOpsBench repository.")

REPO = find_repo()
ARTIFACTS = REPO / "artifacts" / "kdd_tutorial"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print("repo:", REPO)
print("python:", sys.version.split()[0])


In [ ]:
# Load environment variables from .env file in the repository root
from dotenv import load_dotenv

def find_repo(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").is_dir():
            return candidate
    return None
    
repo = find_repo()
if repo is None:
    raise RuntimeError("Open this notebook from inside the AssetOpsBench repository.")
ENV_FILE = repo / ".env"
if not ENV_FILE.exists():
    raise RuntimeError(f"Missing {ENV_FILE}. Complete 00_environment_setup.ipynb first.")
load_dotenv(ENV_FILE, override=True)
print("environment source:", ENV_FILE)

## 1. Create a read-only MCP client

`AOB_READONLY=1` is passed to every newly spawned server process before registration occurs. Consequently, mutation tools such as create, update, approve, assign, close, and cancel are not exposed.


In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

MCP_ENV = os.environ.copy()
MCP_ENV["AOB_READONLY"] = "1"

async def wo_request(operation, tool_name=None, arguments=None):
    params = StdioServerParameters(
        command="uv",
        args=["run", "--directory", str(REPO), "wo-mcp-server"],
        cwd=str(REPO),
        env=MCP_ENV,
    )
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            if operation == "list":
                return await session.list_tools()
            return await session.call_tool(tool_name, arguments or {})

def parse_result(result):
    text = "\n".join(getattr(item, "text", str(item)) for item in result.content)
    try:
        payload = json.loads(text)
    except json.JSONDecodeError:
        payload = text
    if isinstance(payload, dict) and payload.get("error"):
        raise RuntimeError(payload["error"])
    if isinstance(payload, str) and (
        payload.startswith("Unknown tool:") or payload.startswith("Error executing tool")
    ):
        raise RuntimeError(payload)
    return payload

async def list_wo_tools():
    response = await wo_request("list")
    return [
        {"name": tool.name, "description": tool.description, "schema": tool.inputSchema}
        for tool in response.tools
    ]

async def call_wo(name, **arguments):
    return parse_result(await wo_request("call", name, arguments))


## 2. Discover and validate the live read-only contract


In [ ]:
tools = await list_wo_tools()
contract = {
    tool["name"]: list(tool["schema"].get("properties", {}))
    for tool in tools
}
contract


In [ ]:
EXPECTED_READ_TOOLS = {
    "list_workorders",
    "get_workorder",
    "get_workorder_tasks",
    "get_workorder_costs",
    "get_workorder_actuals_vs_planned",
    "get_workorder_kpis",
    "get_schedule_calendar",
    "get_my_assigned_workorders",
    "get_failure_codes",
}
WRITE_TOOLS = {
    "generate_work_order", "update_workorder", "approve_workorder",
    "assign_technician", "close_workorder", "cancel_workorder",
}
assert set(contract) == EXPECTED_READ_TOOLS, (
    f"Read-only contract changed. Missing={sorted(EXPECTED_READ_TOOLS - set(contract))}; "
    f"unexpected={sorted(set(contract) - EXPECTED_READ_TOOLS)}"
)
assert WRITE_TOOLS.isdisjoint(contract), "A write tool is exposed despite AOB_READONLY=1."
print("Read-only work-order contract is compatible: 9 read tools, 0 write tools.")


## 3. List and filter work orders

This first domain query also checks that the seeded `workorder` database is available. `page_size=0` requests all matches, allowing the notebook to select real identifiers rather than rely on placeholders.


In [ ]:
SITE_ID = "MAIN"
try:
    listing = await call_wo(
        "list_workorders", site_id=SITE_ID, page_size=0, page_num=1
    )
except RuntimeError as exc:
    raise RuntimeError(
        "Work-order data is not ready. Complete the CouchDB setup from notebook 02 and load "
        "the default data with: uv run python src/couchdb/init_data.py"
    ) from exc

work_orders = listing.get("work_orders", [])
assert work_orders, f"No work orders are loaded for site {SITE_ID}."
print("work orders found:", listing.get("total"))
[
    {
        "wonum": row.get("wonum"),
        "status": row.get("status"),
        "assetnum": row.get("assetnum"),
        "priority": row.get("wopriority"),
        "description": row.get("description"),
    }
    for row in work_orders
]


In [ ]:
sample = next((row for row in work_orders if row.get("wplabor")), work_orders[0])
WORK_ORDER_NUMBER = str(sample["wonum"])
WORK_ORDER_SITE = str(sample["siteid"])
FILTER_STATUS = sample.get("status")
filtered = await call_wo(
    "list_workorders",
    site_id=WORK_ORDER_SITE,
    status=FILTER_STATUS,
    page_size=10,
    page_num=1,
)
print("selected work order:", WORK_ORDER_NUMBER)
print("status filter:", FILTER_STATUS)
print("matching records:", filtered.get("total"))


## 4. Retrieve one exact work order


In [ ]:
work_order_result = await call_wo(
    "get_workorder", wonum=WORK_ORDER_NUMBER, site_id=WORK_ORDER_SITE
)
work_order = work_order_result.get("work_order", {})
assert str(work_order.get("wonum")) == WORK_ORDER_NUMBER
assert str(work_order.get("siteid")) == WORK_ORDER_SITE
work_order_result


## 5. Inspect child tasks

A work order may have zero or more child tasks. An empty list is a valid grounded result for a parent without task records.


In [ ]:
tasks = await call_wo(
    "get_workorder_tasks", wonum=WORK_ORDER_NUMBER, site_id=WORK_ORDER_SITE
)
tasks


## 6. Compare costs and planned-versus-actual values


In [ ]:
costs = await call_wo(
    "get_workorder_costs", wonum=WORK_ORDER_NUMBER, site_id=WORK_ORDER_SITE
)
actuals_vs_planned = await call_wo(
    "get_workorder_actuals_vs_planned",
    wonum=WORK_ORDER_NUMBER,
    site_id=WORK_ORDER_SITE,
)
{
    "costs": costs,
    "actuals_vs_planned": actuals_vs_planned,
}


## 7. Compute site KPIs

The sample records are historical, so a 120-month window is used to include them. A short current window can correctly return zeros.


In [ ]:
kpis = await call_wo(
    "get_workorder_kpis", site_id=SITE_ID, period_months=120
)
kpis


## 8. Build a historical schedule calendar

Explicit dates match the seeded 2020 tutorial records and avoid an empty calendar caused by a current-date default window.


In [ ]:
schedule = await call_wo(
    "get_schedule_calendar",
    site_id=SITE_ID,
    date_from="2020-01-01",
    date_to="2020-12-31",
    group_by="date",
)
schedule


## 9. Find work assigned to a technician

The labor code is taken from the selected record, keeping the example grounded in the loaded data.


In [ ]:
labor_rows = work_order.get("wplabor") or []
if labor_rows and labor_rows[0].get("laborcode"):
    LABOR_CODE = str(labor_rows[0]["laborcode"])
    assigned = await call_wo(
        "get_my_assigned_workorders",
        labor_code=LABOR_CODE,
        site_id=WORK_ORDER_SITE,
        open_only=True,
    )
    print("technician:", LABOR_CODE)
    display(assigned)
else:
    print("The selected record has no planned labor assignment; example skipped.")


## 10. Read and resolve failure codes

Omitting `code` lists the reference catalog. When the selected work order has a failure-code identifier, the second call resolves that exact code without modifying the record.


In [ ]:
failure_code_catalog = await call_wo("get_failure_codes")
print("failure codes available:", failure_code_catalog.get("total"))
display(failure_code_catalog)

existing_code = work_order.get("failurecode")
if existing_code:
    resolved_failure_code = await call_wo(
        "get_failure_codes", code=str(existing_code)
    )
    print("work-order failure code:", existing_code)
    display(resolved_failure_code)
else:
    print("The selected work order has no recorded failure code.")


## Safety and scope

The server implementation also supports creating, updating, approving, assigning, closing, and cancelling work orders. Those operations are intentionally absent from this tutorial process because `AOB_READONLY=1` prevents their registration. Use a disposable database for lifecycle demonstrations.

## Takeaway

You discovered and exercised the complete read-only work-order MCP contract through real stdio calls while preserving the tutorial database.
